# Lab 10 · The human decides, the council learns

**Prepared by Mustafa Assaf** · 15–20 minute lab

## Put the human in charge, and remember why

**So far:** in Lab 09 the council chose microservices for the Payments platform, 2 votes to 1. The Guardian's dissent, that one operations engineer cannot run many services, was kept but outvoted.

**New:** the two feedback loops of an architecture council:

- **Short loop: a human in the loop.** The human architect reviews each proposal and gives a reason. The council revises with that feedback, and nothing is final until the human accepts it.
- **Long loop: memory.** The accepted decision and the human's reasons are stored in a vector database. For a later case, the council recalls similar past decisions before it proposes.

Then we measure whether memory helps, on *held-out* cases that never gave any feedback. Testing on the feedback case itself would measure memorisation, not learning.

To keep the lab short, `council` stands in for Lab 09's council with one model call. In your project, call your full council there. The memory uses Lab 02's embedding model and Chroma database. The first run downloads the embedding model (about 80 MB).

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Practice answers are scripted. For a live model, save `FREEINFERENCE_API_KEY` in Colab Secrets and change `MODE` to `"live"`. `ask` stops after 12 calls; run this cell again to reset the count.

In [ ]:
from importlib.util import find_spec
if not all(find_spec(name) for name in ['coursekit', 'chromadb']):
    %pip install -q "agentic-ai-coursekit[rag] @ https://github.com/mkassaf/agentic-ai-course/archive/dbe84948102e503e581fd5236bb7e67e63dba56c.zip"
from coursekit import make_ask
MODE = "practice"
ask = make_ask(mode=MODE)

## Step 1 · The proposal and a stand-in council

`proposal` is the chair's outcome from Lab 09. `council` stands in for the whole council: it gives the written policy, any notes, and the case, and asks for one option. It follows the policy unless a note says otherwise, and it rejects any reply that is not an option.

In [ ]:
case = ("Payments platform: 12 developers in 3 squads that release on their own schedules; "
        "one engineer runs operations.")
proposal = {"option": "microservices", "votes": {"microservices": 2, "modular monolith": 1},
            "dissent": "Guardian: One engineer cannot run many services. ['F3']"}
OPTIONS = ["modular monolith", "microservices"]
R1 = "R1: Choose microservices only when the team has 8+ people and services must deploy independently."

def council(case, notes="", example=""):
    """Stands in for the Lab 09 council: one call that returns the proposed option."""
    prompt = (f"Our architecture policy: {R1}\n{notes}\nCase: {case}\n"
              f"Follow the policy unless a note above overrides it. Choose one of {OPTIONS}. Reply with the option only.")
    option = ask(prompt, example=example).strip().rstrip(".").lower()
    if option not in OPTIONS:
        raise ValueError(f"The council replied with an unknown option: {option!r}")
    return option

print(f"Council proposes {proposal['option']} {proposal['votes']}\nDissent: {proposal['dissent']}")

## Step 2 · The human reviews, the council revises

You are the human architect, and `human_review` is your judgement: a verdict and a reason for any option the council proposes. The reason matters most, because it is what the council and the memory learn from. (To review interactively, return the answers of two `input()` calls instead.)

The loop shows each proposal to the human. On a rejection, the reason joins the feedback, and the council proposes again with all the feedback so far. The human's feedback overrides the written policy, because the human keeps the final authority. At most two reviews keep the loop bounded: if the human has not accepted by then, the assert stops the run, and the human decides directly.

In [ ]:
def human_review(option):
    """You are the human architect: a verdict and a reason for each option the council proposes."""
    if option == "microservices":
        return {"verdict": "reject",
                "reason": "One operations engineer cannot run a service per squad; the operating cost is too high."}
    return {"verdict": "accept", "reason": "One deployable with a module per squad is something one engineer can run."}

option, feedback = proposal["option"], ""
for n in (1, 2):  # at most two reviews, so the loop always ends
    review = human_review(option)
    print(f"Review {n}: {option} -> {review['verdict']}: {review['reason']}")
    if review["verdict"] == "accept":
        break
    feedback += f"The human architect rejected {option}: {review['reason']}\n"
    notes = "Feedback from the human architect on this case. It overrides the policy:\n" + feedback
    option = council(case, notes, example="modular monolith")
assert review["verdict"] == "accept", "No proposal was accepted after two reviews; the human decides directly."

## Step 3 · Store the decision in memory

The memory stores the case as the document, so that a new case finds it by meaning, as in Lab 02. The lesson goes into the record's metadata: the human's feedback and the accepted decision. `recall` returns the lessons of past cases within `max_distance`, with their distances. Recalling the stored case itself gives a distance of 0 and prints the lesson.

In [ ]:
import chromadb
from chromadb.config import Settings
from chromadb.utils.embedding_functions import DefaultEmbeddingFunction

db = chromadb.Client(Settings(anonymized_telemetry=False))  # in memory, as in Lab 02
memory = db.get_or_create_collection("decisions", embedding_function=DefaultEmbeddingFunction(),
                                     configuration={"hnsw": {"space": "cosine"}})
lesson = f"Past case: {case}\n{feedback}The human architect accepted {option}: {review['reason']}"
memory.upsert(ids=["D1"], documents=[case], metadatas=[{"lesson": lesson}])

def recall(new_case, max_distance=0.5):
    """The lessons of similar past cases as (distance, lesson) pairs; distant cases are ignored."""
    found = memory.query(query_texts=[new_case], n_results=1)
    pairs = zip(found["distances"][0], found["metadatas"][0])
    return [(round(d, 2), meta["lesson"]) for d, meta in pairs if d <= max_distance]

print(recall(case)[0][1])

## Step 4 · Which new cases recall the lesson?

`held_out` holds two new cases that gave no feedback, with the decision the human architect expects for each. Both recall the Payments lesson. The Search platform is even *closer*, although its four operations engineers make the lesson irrelevant. Similar text does not mean that a lesson applies. That is why the lesson keeps its facts and its reason, and why the prompt in Step 5 asks to apply a lesson only when its facts match.

In [ ]:
held_out = [  # new cases, never used for feedback, and the decision the human architect expects
    ("Logistics tracker: 10 developers in 2 squads that release on their own schedules; "
     "one engineer runs operations.", "modular monolith"),
    ("Search platform: 15 developers in 3 squads that release on their own schedules; "
     "four engineers run operations.", "microservices"),
]
for c, _ in held_out:
    print("recalled at distance", [d for d, _ in recall(c)], "by", c)

## Step 5 · Score the held-out cases before and after memory

Each held-out case is answered twice: without memory, when the council follows the written policy, and with the recalled lessons. In practice mode, `scripted` holds the council's replies for each run. Two cases only demonstrate the measurement; the course project asks for a benchmark of 20 or more scenarios.

In [ ]:
assert case not in [c for c, _ in held_out], "Never test on a case that gave feedback."
scripted = {"before": ["microservices", "microservices"], "after": ["modular monolith", "microservices"]}
scores = {}
for run in ("before", "after"):
    answers = []
    for (c, expected), example in zip(held_out, scripted[run]):
        lessons = [text for _, text in recall(c)] if run == "after" else []
        notes = ("Lessons from the human architect's past decisions. They override the policy, "
                 "but only when their facts match this case:\n" + "\n".join(lessons)) if lessons else ""
        answers.append(council(c, notes, example=example))
    scores[run] = sum(a == expected for a, (_, expected) in zip(answers, held_out))
    print(f"{run:<6} memory: {answers} -> {scores[run]}/{len(held_out)} match the human architect")
assert scores["after"] > scores["before"], "Memory did not improve the held-out score; compare the answers."

## What you should see

Step 1 prints the council's proposal and the Guardian's dissent. In Step 2, review 1 rejects microservices, the council revises to the modular monolith, and review 2 accepts it. Step 3 prints the stored lesson. Step 4 prints a distance of about 0.33 for the Logistics tracker and about 0.22 for the Search platform. In Step 5, without memory both cases get microservices, so 1 of 2 matches the human architect. With memory, the Logistics tracker gets the modular monolith and the Search platform keeps microservices: 2 of 2.

The human's reason did the work: the review that changed the Payments decision also changed a new case, without a new rule.

## Try it

1. Accept the first proposal: make `human_review` accept microservices. The council never revises, and the memory stores microservices as the accepted decision.
2. Make `human_review` reject every option. The loop ends after two reviews, and the assert hands the decision to you.
3. Set `max_distance=0.2` in `recall` and run Steps 3 and 4. Neither held-out case recalls the lesson now. In live mode, what happens to the score after memory?
4. Add a held-out case with one operations engineer that should still get microservices, for example because a platform team runs the services for it. Does the lesson overreach? What would you add to the lesson?
5. In live mode, does the model apply the lesson to the Logistics tracker and leave the Search platform unchanged? Run Step 5 a few times. A live model does not apply a lesson every time, which is why the course project measures many scenarios over repeated runs.

## Takeaway

The human architect keeps the final decision: the council revises with the human's reasons until the human accepts. Memory turns those reasons into lessons for later cases, and retrieval finds them by similarity. Whether a lesson applies is a separate question, so keep its facts and its reason with it. Measure memory on held-out cases, never on the cases that gave the feedback. You have now built every part of an architecture council: chained steps, independent proposals, challenges, a chair, a human, and a memory. Lab 11 decides which cases need the council at all, and Lab 12 tests whether it beats a single model.

## Read further

- [Shinn et al., Reflexion: language agents that learn from verbal feedback (2023)](https://arxiv.org/abs/2303.11366)
- [Park et al., Generative agents: memory and retrieval (2023)](https://arxiv.org/abs/2304.03442)